In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("EIA_API_KEY")
print(len(key))

In [ ]:
url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

params = {
    "api_key": key,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "CISO",
    "facets[type][]": "D",
    "start": "2026-09-01T00",
    "end": "2026-09-01T23"
}

response = requests.get(url, params=params)

In [ ]:
#INSPECt

#status
if response.status_code == 200:
    print(response.status_code)
    #shape
    payload = response.json()
    print(payload.keys())
    rows = payload["response"]["data"]
    print(len(rows))
    print(rows[0])
else:
    print("Request failed:", response.status_code)
    print(response.text[:500])


In [ ]:
#dataframe
import pandas as pd

df = pd.DataFrame(rows)

df["value"] = pd.to_numeric(df["value"])
df["period"] = pd.to_datetime(df["period"], utc=True)

df = df.sort_values("period")  # put hours in order
df.reset_index(drop=True)
print(df[["period", "type", "value"]].head(24))
print(df.dtypes)

In [ ]:
df["period_pt"] = df["period"].dt.tz_convert("America/Los_Angeles") #converts timezone
df["hour_pt"] = df["period_pt"].dt.hour

peak = df.loc[df["value"].idxmax()] # row with the max value
print(peak[["period_pt", "hour_pt", "value"]])

In [ ]:
#pulling solar

fuel_url = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"

fuel_params = {
    "api_key": key,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "CISO",
    "facets[fueltype][]": ["SUN", "WND"],  
    "start": "2026-09-01T00",
    "end": "2026-09-01T23",
}

fuel_resp = requests.get(fuel_url, params=fuel_params, timeout=30)
print(fuel_resp.status_code)
fuel_rows = fuel_resp.json()["response"]["data"]
print(len(fuel_rows))
print(fuel_rows[0])

In [ ]:
fuel = pd.DataFrame(fuel_rows)
fuel["value"] = pd.to_numeric(fuel["value"])
fuel["period"] = pd.to_datetime(fuel["period"], utc=True)

# making it long → wide: one row per hour, one column per fuel, making it better for storing
wide = fuel.pivot(index="period", columns="fueltype", values="value")

# attach demand by matching timestamps, join matches on index, merge matches on column
grid = wide.join(df.set_index("period")["value"].rename("demand"), how="inner")

grid["renewable"] = grid["SUN"] + grid["WND"]
grid["net_load"] = grid["demand"] - grid["renewable"]

grid.index = grid.index.tz_convert("America/Los_Angeles")
print(grid)

In [80]:
grid.describe()

,SUN,WND,demand,renewable,net_load
count,24.000000,24.000000,24.000000,24.000000,24.000000
mean,8336.000000,2917.083333,30534.791667,11253.083333,19281.708333
std,9261.725292,1005.898902,3817.311232,9080.141849,7278.699911
min,-64.000000,1653.000000,24535.000000,1971.000000,9196.000000
25%,-60.000000,1913.750000,26880.750000,3031.750000,11973.750000
50%,2536.500000,3002.500000,32281.500000,6798.500000,22011.500000
75%,19174.000000,3443.250000,33255.750000,21842.500000,24504.000000
max,20471.000000,4762.000000,35589.000000,23261.000000,30042.000000
